In [1]:
# Notebook parameters (modify as needed)#
model_name = "facebook/opt-1.3b"
#model_name = "huggyllama/llama-7b"
prompt = "What is steganography?"
m = 80            # length of generation
n = 256           # watermark sequence length
key = 42          # watermark key
seed = 0          # random seed for reproducibility

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device(
    'cuda:0' if torch.cuda.is_available() else
    ('mps' if torch.backends.mps.is_available() else 'cpu')
)

In [3]:
# Seed for reproducibility
torch.manual_seed(seed)

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Prepare inputs
inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

# Generate normally
outputs = model.generate(
    inputs,
    max_new_tokens=m,
    do_sample=True,
    temperature=1.0,
)

# Decode and display
normal_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Normal Output ---")
print(normal_text)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


--- Normal Output ---
What is steganography?

Steganography is a means of secret communication by means of encryption using another encryption system, typically one not known to the recipient. The secret key used in stegography (also called ‘steganography’) consists of several pieces of information, including the message to be encrypted, and the key itself. These pieces are typically stored in a hidden text file or image file.


In [4]:
# Mersenne Twister based RNG for watermarking
def int_32(x): return int(0xFFFFFFFF & x)

class mersenne_rng:
    def __init__(self, seed=5489):
        self.state = [0]*624
        self.index = 624
        self.upper_mask = (1<<31)
        self.lower_mask = (1<<31) - 1
        # constants
        self.f, self.m = 1812433253, 397
        self.u, self.s, self.b = 11, 7, 0x9D2C5680
        self.t, self.c, self.l = 15, 0xEFC60000, 18
        # initialize state
        self.state[0] = seed
        for i in range(1, 624):
            prev = self.state[i-1]
            self.state[i] = int_32(self.f * (prev ^ (prev >> 30)) + i)

    def twist(self):
        for i in range(624):
            x = (self.state[i] & self.upper_mask) + (self.state[(i+1)%624] & self.lower_mask)
            xA = x >> 1
            if x % 2: xA ^= 0x9908b0df
            self.state[i] = self.state[(i + self.m) % 624] ^ xA
        self.index = 0

    def rand(self):
        if self.index >= 624:
            self.twist()
        y = self.state[self.index]
        y ^= (y >> self.u)
        y ^= ((y << self.s) & self.b)
        y ^= ((y << self.t) & self.c)
        y ^= (y >> self.l)
        self.index += 1
        return (int_32(y) * (1.0 / 4294967296.0))

# Exponential watermark sampling
def exp_sampling(probs, u):
    # probs: tensor [vocab], u: numpy array [vocab]
    return torch.argmax(torch.from_numpy(u) ** (1.0 / probs.cpu().numpy()), dim=1).unsqueeze(-1)

# Watermark generation
import numpy as np

def generate_shift(model, prompt_ids, vocab_size, n, m, key):
    rng = mersenne_rng(key)
    # create watermark matrix xi
    xi = np.array([rng.rand() for _ in range(n * vocab_size)], dtype=np.float32).reshape(n, vocab_size)
    shift = torch.randint(0, n, (1,)).item()

    inputs = prompt_ids.to(model.device)
    attn = torch.ones_like(inputs)
    past = None
    for i in range(m):
        with torch.no_grad():
            if past:
                out = model(inputs[:, -1:], past_key_values=past, attention_mask=attn)
            else:
                out = model(inputs)
        probs = torch.nn.functional.softmax(out.logits[:, -1, :vocab_size], dim=-1)
        u = xi[(shift + i) % n]
        token = exp_sampling(probs, u)
        inputs = torch.cat([inputs, token.to(model.device)], dim=-1)
        past = out.past_key_values
        attn = torch.cat([attn, torch.ones((attn.shape[0],1), device=attn.device)], dim=-1)

    return inputs.cpu()

In [5]:
# Encode prompt & run watermark generation
tokens = tokenizer.encode(prompt, return_tensors='pt')
wm_tokens = generate_shift(model, tokens, len(tokenizer), n, m, key)[0]
watermarked_text = tokenizer.decode(wm_tokens, skip_special_tokens=True)
print("--- Watermarked Output ---")
print(watermarked_text)

<ipython-input-4-41e7f3e70fa0>:42: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return torch.argmax(torch.from_numpy(u) ** (1.0 / probs.cpu().numpy()), dim=1).unsqueeze(-1)


--- Watermarked Output ---
What is steganography?

Steganography is a clandestine erase of data information that can range anywhere from documents, emails, pieces of paper, to anything that can be secreted or erased. Whether embedded into the lock of a door or hidden in a wallpaper defacement, to replicate electronically Stego.l’s techniques are agile and are enhanced by adaptive algorithms and cloud-based storage. The digital/


In [22]:
from numba import njit, prange
import numpy as np
import math

@njit
def levenshtein(x, y, gamma=0.0):
    n = len(x)
    m = len(y)
    A = np.zeros((n + 1, m + 1), dtype=np.float64)
    for i in range(n + 1):
        for j in range(m + 1):
            if i == 0:
                A[i, j] = j * gamma
            elif j == 0:
                A[i, j] = i * gamma
            else:
                cost_sub = math.log(1.0 - y[j - 1, x[i - 1]])
                # deletion
                best = A[i - 1, j] + gamma
                # insertion
                tmp = A[i, j - 1] + gamma
                if tmp < best:
                    best = tmp
                # substitution
                tmp = A[i - 1, j - 1] + cost_sub
                if tmp < best:
                    best = tmp
                A[i, j] = best
    return A[n, m]


In [23]:
import numpy as np
import time
import tqdm

from numba import njit, prange


@njit
def detect(tokens, n, k, xi, gamma=0.0):
    m = len(tokens)
    n = len(xi)

    A = np.empty((m - (k - 1), n))
    for i in range(m - (k - 1)):
        for j in range(n):
            A[i][j] = levenshtein(tokens[i : i + k], xi[(j + np.arange(k)) % n], gamma)

    return np.min(A)

def permutation_test(tokens, key, n, k, vocab_size, n_runs=100):
    rng = mersenne_rng(key)
    xi = np.array(
        [rng.rand() for _ in range(n * vocab_size)], dtype=np.float32
    ).reshape(n, vocab_size)
    test_result = detect(tokens, n, k, xi)

    p_val = 0
    for run in tqdm.tqdm(range(n_runs)):
        xi_alternative = np.random.rand(n, vocab_size).astype(np.float32)
        null_result = detect(tokens, n, k, xi_alternative)

        # assuming lower test values indicate presence of watermark
        p_val += null_result <= test_result

    return (p_val + 1.0) / (n_runs + 1.0)


# Run decoding test
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer.encode(watermarked_text, return_tensors='pt', truncation=True, max_length=2048).numpy()[0]
tok_list = wm_tokens.cpu().numpy().tolist()
print(tok_list)
start = time.time()
p_val = permutation_test(tok_list, key, n, len(tok_list), len(tokenizer))
print()
print(f"p-value: {p_val:.4f} (elapsed {time.time()-start:.1f}s)")
print(f"Is watermarked: {p_val<0.01}")

[2, 2264, 16, 11235, 3494, 10486, 116, 50118, 50118, 21426, 3494, 10486, 16, 10, 35146, 24300, 9, 414, 335, 14, 64, 1186, 4558, 31, 2339, 6, 5575, 6, 3745, 9, 2225, 6, 7, 932, 14, 64, 28, 3556, 196, 50, 23717, 4, 5994, 14224, 88, 5, 7014, 9, 10, 1883, 50, 7397, 11, 10, 37103, 3816, 23423, 6, 7, 19795, 30319, 312, 3733, 139, 4, 462, 17, 27, 29, 7373, 32, 28144, 8, 32, 9094, 30, 28760, 16964, 8, 3613, 12, 805, 3521, 4, 20, 1778, 73]


100%|██████████| 100/100 [01:55<00:00,  1.16s/it]


p-value: 0.0099 (elapsed 138.7s)
Is watermarked: True
